In [0]:
%pip install \
langchain==0.2.16 \
langgraph==0.2.16 \
langchain-community==0.2.16 \
langchain-groq==0.1.9 \
faiss-cpu==1.8.0 \
sentence-transformers==3.0.1 \
pandas==2.2.2 \
numpy==1.26.4 \
pydantic==2.8.2 \
rapidfuzz==3.9.6 \
beautifulsoup4==4.12.3 \
requests==2.32.3 \
geopy==2.4.1 \
folium==0.17.0 \
mlflow

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import json
import os
import re
import requests
import warnings

from bs4 import BeautifulSoup
from rapidfuzz import fuzz
from geopy.distance import geodesic

from typing import Dict, List
from typing_extensions import TypedDict

from langchain_groq import ChatGroq
from langchain.schema import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")

In [0]:
from facility_and_ngo_fields import *
from free_form import *
from medical_specialties import *
from organization_extraction import *

In [0]:
df = pd.read_csv(
    "/Volumes/workspace/default/my_volume/Virtue Foundation Ghana v0.3 - Sheet1.csv"
)

df.fillna("", inplace=True)

print(df.shape)
df.head()

(987, 41)


,source_url,name,pk_unique_id,mongo DB,specialties,procedure,equipment,capability,organization_type,content_table_id,phone_numbers,email,websites,officialWebsite,yearEstablished,acceptsVolunteers,facebookLink,twitterLink,linkedinLink,instagramLink,logo,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,countries,missionStatement,missionStatementLink,organizationDescription,facilityTypeId,operatorTypeId,affiliationTypeIds,description,area,numberDoctors,capacity,unique_id
0,https://www.linkedin.com/company/waaf/,109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...,1,62aa51490990af00169ab9ed,"[""infectiousDiseases"",""maternalFetalMedicineOr...",,,"[""Has a location at 109/No 1 Bekwai Rd (Near M...",facility,a77400f4-6203-4b0d-84ad-fd143bd768e3,"[""+233249354576"",""+233203928883""]",,"[""waafweb.org""]",waafweb.org,,,,,,,,109/No 1 Bekwai Rd (Near Mexico Hotel),,,Takoradi,,,Ghana,GH,,,,,clinic,,,"WAAF is committed to battling HIV/AIDS, TB, an...",,,,7d362eaa-8130-410a-bf23-7be549177af0
1,https://www.ghanabusinessweb.com/accra-adabrak...,1st Foundation Clinic,2,,"[""internalMedicine""]",[],[],"[""Located in Dansoman, Accra, Ghana, opposite ...",facility,cd191c26-2987-404f-b5bb-7dfe6d7a7b02,,,,,,,,,,,,Opp. Standard Chartered Bank,,,Dansoman,,,Ghana,GH,,,,,clinic,,,,,,,9d70e24f-247c-41af-b708-cc10e99e54b1
2,https://www.ghanabusinessweb.com/accra-cantonm...,1st Foundation Clinic,2,,"[""internalMedicine""]",[],[],[],facility,6cc7060e-63f3-4e20-b83c-483ac1c3206e,,,,,,,,,,,,"Opp. Standard Chartered Bank, Dansoman",,,Accra,,,Ghana,GH,,,,,clinic,,,,,,,97d11408-8303-44c3-83c2-72cb91c58fb1
3,https://www.ghanabusinessweb.com/accra-dansoma...,1st Foundation Clinic,2,,"[""internalMedicine""]",[],[],[],facility,2d84a935-452c-441c-a766-8fcbb8b4e7ea,,,,,,,,,,,,Opp. Standard Chartered Bank,Dansoman,,Accra,,,Ghana,GH,,,,,clinic,,,,,,,4c17951e-87e0-4472-8cf4-189cea9782b8
4,https://www.ghanabusinessweb.com/accra-osu-hea...,1st Foundation Clinic,2,,"[""internalMedicine""]",,,,facility,78038e2e-3210-4a0b-a502-ad0f8aabbb38,,,,,,,,,,,,Opp. Standard Chartered Bank,Dansoman,,Accra,,,Ghana,GH,,,,,clinic,,,,,,,a6ec226d-a88e-4366-b390-5c709ef54e92


In [0]:
df["master_text"] = (
    df.astype(str).agg(" ".join, axis=1)
)

In [0]:
GROQ_API_KEY = "YOUR_API_KEY"

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [0]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
documents = []

for idx, row in df.iterrows():

    text = row["master_text"]

    metadata = {
        "row_id": idx,
        "facility": row.get("name", ""),
        "city": row.get("address_city", ""),
        "country": row.get("address_country", ""),
        "specialties": row.get("specialties", ""),
        "capability": row.get("capability", ""),
        "equipment": row.get("equipment", "")
    }

    documents.append(
        Document(
            page_content=text,
            metadata=metadata
        )
    )

print(len(documents))

987


In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(documents)

print(len(split_docs))

1196


In [0]:
vectorstore = FAISS.from_documents(
    split_docs,
    embeddings
)

In [0]:
REQUIRED_INFRASTRUCTURE = {
    "icu": ["ventilator", "oxygen", "critical care"],
    "cardiac surgery": ["operating room", "icu"],
    "mri diagnostics": ["mri"],
    "dialysis": ["dialysis machine"],
    "trauma center": ["operating room", "blood bank"],
    "cataract surgery": ["operating microscope"],
    "nicu": ["neonatal ventilator"],
    "stroke care": ["ct scanner"],
    "neurosurgery": ["icu", "operating room"],
}

In [0]:
def validate_capabilities(row):

    text = str(row["master_text"]).lower()

    validations = []

    for capability, required_items in REQUIRED_INFRASTRUCTURE.items():

        if capability in text:

            missing = []

            for item in required_items:
                if item not in text:
                    missing.append(item)

            validations.append({
                "capability": capability,
                "required": required_items,
                "missing": missing,
                "supported": len(missing) == 0,
                "confidence": round(
                    1 - (len(missing) / len(required_items)),
                    2
                )
            })

    return validations

In [0]:
df["capability_validation"] = df.apply(
    validate_capabilities,
    axis=1
)

print(df[["name", "capability_validation"]].head(30))

                                                 name                              capability_validation
0   109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...                                                 []
1                               1st Foundation Clinic                                                 []
2                               1st Foundation Clinic                                                 []
3                               1st Foundation Clinic                                                 []
4                               1st Foundation Clinic                                                 []
5                               2BN Military Hospital                                                 []
6                                37 Military Hospital                                                 []
7                                37 Military Hospital                                                 []
8                    3E Medical Center - Accra, Ghana  

In [0]:
def safe_int(value):

    try:

        if pd.isna(value):
            return 0

        value = str(value).strip()

        if value == "":
            return 0

        return int(float(value))

    except:
        return 0


def calculate_strength_score(row):

    score = 0
    reasons = []

    # ----------------------------
    # SAFE FIELD EXTRACTION
    # ----------------------------

    doctors = safe_int(row.get("numberDoctors", 0))
    capacity = safe_int(row.get("capacity", 0))
    area = safe_int(row.get("area", 0))

    equipment_text = str(row.get("equipment", "")).lower()
    capability_text = str(row.get("capability", "")).lower()
    specialties_text = str(row.get("specialties", "")).lower()
    procedures_text = str(row.get("procedure", "")).lower()

    website = str(row.get("officialWebsite", "")).strip()

    # ----------------------------
    # DOCTOR SCORE
    # ----------------------------

    if doctors >= 50:
        score += 25
        reasons.append("High doctor count")

    elif doctors >= 15:
        score += 18
        reasons.append("Moderate doctor count")

    elif doctors >= 5:
        score += 10
        reasons.append("Basic doctor coverage")

    else:
        score += 2
        reasons.append("Very low doctor count")

    # ----------------------------
    # CAPACITY SCORE
    # ----------------------------

    if capacity >= 300:
        score += 20
        reasons.append("Large hospital capacity")

    elif capacity >= 100:
        score += 15
        reasons.append("Moderate hospital capacity")

    elif capacity >= 20:
        score += 8
        reasons.append("Small inpatient facility")

    # ----------------------------
    # AREA SCORE
    # ----------------------------

    if area >= 10000:
        score += 10
        reasons.append("Large infrastructure footprint")

    elif area >= 3000:
        score += 5
        reasons.append("Moderate infrastructure footprint")

    # ----------------------------
    # ICU CAPABILITY
    # ----------------------------

    if "icu" in capability_text:
        score += 15
        reasons.append("ICU capability detected")

    # ----------------------------
    # ADVANCED EQUIPMENT
    # ----------------------------

    advanced_equipment = [
        "mri",
        "ct",
        "ventilator",
        "dialysis",
        "operating microscope",
        "x-ray"
    ]

    detected_equipment = []

    for item in advanced_equipment:

        if item in equipment_text:
            detected_equipment.append(item)

    equipment_score = len(detected_equipment) * 4

    score += equipment_score

    if detected_equipment:
        reasons.append(
            f"Advanced equipment detected: {', '.join(detected_equipment)}"
        )

    # ----------------------------
    # SPECIALTY DEPTH
    # ----------------------------

    specialty_count = len(
        [x for x in specialties_text.split(",") if x.strip()]
    )

    score += min(specialty_count * 2, 10)

    if specialty_count >= 4:
        reasons.append("Multiple specialties available")

    # ----------------------------
    # DIGITAL PRESENCE
    # ----------------------------

    if website != "":
        score += 5
        reasons.append("Official website available")

    # ----------------------------
    # SURGICAL SUPPORT
    # ----------------------------

    if "surgery" in procedures_text:
        score += 8
        reasons.append("Surgical procedures detected")

    # ----------------------------
    # FINAL CATEGORY
    # ----------------------------

    if score >= 75:
        category = "Strong"

    elif score >= 45:
        category = "Moderate"

    else:
        category = "Weak"

    return {
        "score": score,
        "category": category,
        "reasons": reasons,
        "doctor_count": doctors,
        "capacity": capacity
    }

In [0]:
df["strength_analysis"] = df.apply(
    calculate_strength_score,
    axis=1
)

print(
    df[
        [
            "name",
            "strength_analysis"
        ]
    ].head()
)

                                                name                                  strength_analysis
0  109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...  {'score': 17, 'category': 'Weak', 'reasons': [...
1                              1st Foundation Clinic  {'score': 4, 'category': 'Weak', 'reasons': ['...
2                              1st Foundation Clinic  {'score': 4, 'category': 'Weak', 'reasons': ['...
3                              1st Foundation Clinic  {'score': 4, 'category': 'Weak', 'reasons': ['...
4                              1st Foundation Clinic  {'score': 4, 'category': 'Weak', 'reasons': ['...


In [0]:
CONTRADICTION_RULES = [

    {
        "claim": "neurosurgery",
        "required": ["icu", "operating room"]
    },

    {
        "claim": "cardiology",
        "required": ["ecg", "echo"]
    },

    {
        "claim": "dialysis",
        "required": ["dialysis machine"]
    },

    {
        "claim": "trauma",
        "required": ["blood bank"]
    },

    {
        "claim": "icu",
        "required": ["ventilator", "oxygen"]
    },

    {
        "claim": "cataract surgery",
        "required": ["operating microscope"]
    }
]

In [0]:
def detect_contradictions(row):

    text = str(row["master_text"]).lower()

    contradictions = []

    for rule in CONTRADICTION_RULES:

        claim = rule["claim"]

        if claim in text:

            missing_items = []

            for item in rule["required"]:

                if item not in text:
                    missing_items.append(item)

            if len(missing_items) > 0:

                contradictions.append({

                    "claim": claim,

                    "missing_support": missing_items,

                    "severity": "high"

                })

    return contradictions

In [0]:
df["contradictions"] = df.apply(
    detect_contradictions,
    axis=1
)

print(
    df[
        [
            "name",
            "contradictions"
        ]
    ].head()
)

                                                name contradictions
0  109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...             []
1                              1st Foundation Clinic             []
2                              1st Foundation Clinic             []
3                              1st Foundation Clinic             []
4                              1st Foundation Clinic             []


In [0]:
def enrich_missing_fields(row):

    enrichment = {}

    # ----------------------------
    # WEBSITE ENRICHMENT
    # ----------------------------

    if str(row.get("officialWebsite", "")).strip() == "":

        facility_name = str(row.get("name", "")).strip()

        if facility_name != "":

            inferred_domain = (
                facility_name
                .lower()
                .replace(" ", "")
                .replace(",", "")
                .replace(".", "")
            )

            enrichment["possibleWebsite"] = (
                f"www.{inferred_domain}.org"
            )

    # ----------------------------
    # COUNTRY CODE ENRICHMENT
    # ----------------------------

    country = str(
        row.get("address_country", "")
    ).lower()

    if (
        str(row.get("address_countryCode", "")).strip() == ""
    ):

        mapping = {

            "ghana": "GH",
            "india": "IN",
            "kenya": "KE",
            "uganda": "UG",
            "nigeria": "NG"
        }

        if country in mapping:
            enrichment["predictedCountryCode"] = mapping[country]

    # ----------------------------
    # FACILITY TYPE ENRICHMENT
    # ----------------------------

    facility_type = str(
        row.get("facilityTypeId", "")
    ).strip()

    if facility_type == "":

        text = str(row["master_text"]).lower()

        if "hospital" in text:
            enrichment["predictedFacilityType"] = "hospital"

        elif "clinic" in text:
            enrichment["predictedFacilityType"] = "clinic"

    return enrichment

In [0]:
df["enrichment"] = df.apply(
    enrich_missing_fields,
    axis=1
)

print(
    df[
        [
            "name",
            "enrichment"
        ]
    ].head()
)

                                                name                                         enrichment
0  109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...                                                 {}
1                              1st Foundation Clinic  {'possibleWebsite': 'www.1stfoundationclinic.o...
2                              1st Foundation Clinic  {'possibleWebsite': 'www.1stfoundationclinic.o...
3                              1st Foundation Clinic  {'possibleWebsite': 'www.1stfoundationclinic.o...
4                              1st Foundation Clinic  {'possibleWebsite': 'www.1stfoundationclinic.o...


In [0]:
def validate_capabilities(row):

    text = str(row["master_text"]).lower()

    validations = []

    for capability, required_items in REQUIRED_INFRASTRUCTURE.items():

        if capability in text:

            missing = []

            for item in required_items:

                if item not in text:
                    missing.append(item)

            confidence = round(
                1 - (
                    len(missing) / len(required_items)
                ),
                2
            )

            validations.append({

                "capability": capability,

                "required": required_items,

                "missing": missing,

                "supported": len(missing) == 0,

                "confidence": confidence
            })

    return validations

In [0]:
df["capability_validation"] = df.apply(
    validate_capabilities,
    axis=1
)

print(
    df[
        [
            "name",
            "capability_validation"
        ]
    ].head()
)

                                                name capability_validation
0  109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...                    []
1                              1st Foundation Clinic                    []
2                              1st Foundation Clinic                    []
3                              1st Foundation Clinic                    []
4                              1st Foundation Clinic                    []


In [0]:
def generate_reasoning_trace(row):

    traces = []

    strength = row.get(
        "strength_analysis",
        {}
    )

    traces.append(
        f"Facility classified as {strength.get('category', 'Unknown')}"
    )

    for reason in strength.get("reasons", []):

        traces.append(reason)

    validations = row.get(
        "capability_validation",
        []
    )

    for val in validations:

        if val["supported"]:

            traces.append(
                f"{val['capability']} capability validated"
            )

        else:

            traces.append(
                f"{val['capability']} missing {val['missing']}"
            )

    contradictions = row.get(
        "contradictions",
        []
    )

    for contradiction in contradictions:

        traces.append(

            f"Potential contradiction: "

            f"{contradiction['claim']} "

            f"without {contradiction['missing_support']}"
        )

    return traces

In [0]:
df["reasoning_trace"] = df.apply(
    generate_reasoning_trace,
    axis=1
)

print(
    df[
        [
            "name",
            "reasoning_trace"
        ]
    ].head()
)

                                                name                                    reasoning_trace
0  109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...  [Facility classified as Weak, Very low doctor ...
1                              1st Foundation Clinic  [Facility classified as Weak, Very low doctor ...
2                              1st Foundation Clinic  [Facility classified as Weak, Very low doctor ...
3                              1st Foundation Clinic  [Facility classified as Weak, Very low doctor ...
4                              1st Foundation Clinic  [Facility classified as Weak, Very low doctor ...


In [0]:
def retrieve_facilities(query, k=5):

    docs = vectorstore.similarity_search(
        query,
        k=k
    )

    results = []

    for doc in docs:

        row_id = doc.metadata["row_id"]

        row = df.iloc[row_id]

        results.append({

            "facility": row.get("name", "Unknown"),

            "city": row.get("address_city", ""),

            "strength": row.get(
                "strength_analysis",
                {}
            ),

            "contradictions": row.get(
                "contradictions",
                []
            ),

            "reasoning_trace": row.get(
                "reasoning_trace",
                []
            )
        })

    return results

In [0]:
def find_weak_clinics():

    weak = []

    for _, row in df.iterrows():

        analysis = row.get(
            "strength_analysis",
            {}
        )

        if analysis.get("category") == "Weak":

            weak.append({

                "facility": row.get("name", ""),

                "score": analysis.get("score", 0),

                "reasons": analysis.get("reasons", []),

                "trace": row.get(
                    "reasoning_trace",
                    []
                )
            })

    return weak

In [0]:
def detect_medical_deserts():

    deserts = []

    grouped = df.groupby("address_city")

    for city, rows in grouped:

        city_text = " ".join(
            rows["master_text"].astype(str)
        ).lower()

        if "icu" not in city_text:

            deserts.append({

                "city": city,

                "issue": "No ICU capability detected",

                "facility_count": len(rows)
            })

    return deserts

In [0]:
def idp_agent_llm(query):

    query_lower = query.lower()

    # ----------------------------
    # MEDICAL DESERTS
    # ----------------------------

    if "medical desert" in query_lower:

        return detect_medical_deserts()

    # ----------------------------
    # WEAK CLINICS
    # ----------------------------

    elif "weak clinic" in query_lower:

        return find_weak_clinics()

    # ----------------------------
    # MISSING ICU
    # ----------------------------

    elif "missing icu" in query_lower:

        results = []

        for _, row in df.iterrows():

            capability_text = str(
                row.get("capability", "")
            ).lower()

            if "icu" not in capability_text:

                results.append({

                    "facility": row.get("name", ""),

                    "city": row.get(
                        "address_city",
                        ""
                    )
                })

        return results

    # ----------------------------
    # GENERAL QUERY
    # ----------------------------

    else:

        return retrieve_facilities(query)

In [0]:
print(
    idp_agent_llm(
        "Show hospitals with missing ICU"
    )
)

[{'facility': '109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana', 'city': 'Takoradi'}, {'facility': '1st Foundation Clinic', 'city': 'Dansoman'}, {'facility': '1st Foundation Clinic', 'city': 'Accra'}, {'facility': '1st Foundation Clinic', 'city': 'Accra'}, {'facility': '1st Foundation Clinic', 'city': 'Accra'}, {'facility': '2BN Military Hospital', 'city': 'Apremdo'}, {'facility': '37 Military Hospital', 'city': 'Accra'}, {'facility': '37 Military Hospital', 'city': 'Accra'}, {'facility': '3E Medical Center - Accra, Ghana', 'city': 'Accra'}, {'facility': '3Way Family Care Clinic', 'city': 'Acherensua'}, {'facility': 'A & A Medlove Medical Centre', 'city': 'Accra'}, {'facility': 'A&a Medlove Medical Centre', 'city': 'Accra'}, {'facility': 'Abomosu Health Centre', 'city': 'Abomosu'}, {'facility': 'Aboraa Hospital', 'city': 'Accra'}, {'facility': 'Abuakwa Maternity Home', 'city': 'Abuakwa'}, {'facility': 'Abura Health Centre', 'city': 'Abura'}, {'facility': 'Accra Medical Centre', 

In [0]:
print(
    idp_agent_llm(
        "Which areas are medical deserts?"
    )
)

[{'city': '', 'issue': 'No ICU capability detected', 'facility_count': 64}, {'city': 'ACCRA', 'issue': 'No ICU capability detected', 'facility_count': 2}, {'city': 'AGROYESUM', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'ANOLGA', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Abelenkpe, Accra', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Abesim', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Abesim - Sunyani', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Aboadze', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Abomosu', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Abuakwa', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Abura', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Accra Central', 'issue': 'No ICU capability detected', 'facility_count': 1}, {'city': 'Acc

In [0]:
print(
    idp_agent_llm(
        "Find weak clinics"
    )
)

[{'facility': '109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana', 'score': 17, 'reasons': ['Very low doctor count', 'Multiple specialties available', 'Official website available'], 'trace': ['Facility classified as Weak', 'Very low doctor count', 'Multiple specialties available', 'Official website available']}, {'facility': '1st Foundation Clinic', 'score': 4, 'reasons': ['Very low doctor count'], 'trace': ['Facility classified as Weak', 'Very low doctor count']}, {'facility': '1st Foundation Clinic', 'score': 4, 'reasons': ['Very low doctor count'], 'trace': ['Facility classified as Weak', 'Very low doctor count']}, {'facility': '1st Foundation Clinic', 'score': 4, 'reasons': ['Very low doctor count'], 'trace': ['Facility classified as Weak', 'Very low doctor count']}, {'facility': '1st Foundation Clinic', 'score': 4, 'reasons': ['Very low doctor count'], 'trace': ['Facility classified as Weak', 'Very low doctor count']}, {'facility': '2BN Military Hospital', 'score': 8, 'reason

In [0]:
df.to_csv(
    "/Volumes/workspace/default/my_volume/enhanced_healthcare_dataset.csv",
    index=False
)

In [0]:
vectorstore.save_local(
    "/Volumes/workspace/default/my_volume/healthcare_vectorstore"
)

In [0]:
SYSTEM_PROMPT = """
You are an advanced Healthcare IDP Agent.

Your job is to analyze healthcare facility data and answer questions
about:

- medical infrastructure
- hospital capabilities
- weak clinics
- medical deserts
- contradictions in claims
- specialty availability
- healthcare resource gaps
- suspicious capability claims

You MUST:
- use the provided facility context
- use reasoning traces when useful
- explain contradictions clearly
- avoid hallucinating unsupported facts
- give concise but intelligent answers

If evidence is weak or contradictory,
explicitly mention uncertainty.
"""

In [0]:
def compress_result(result):

    strength = result.get("strength", {})

    contradictions = result.get(
        "contradictions",
        []
    )

    reasoning = result.get(
        "reasoning_trace",
        []
    )

    compressed = {

        "facility": result.get("facility", ""),

        "city": result.get("city", ""),

        "score": strength.get("score", 0),

        "category": strength.get("category", ""),

        "top_reasons": strength.get(
            "reasons",
            []
        )[:3],

        "contradiction_count": len(
            contradictions
        ),

        "important_trace": reasoning[:3]
    }

    return compressed

In [0]:
def build_small_context(results):

    compressed_results = []

    for r in results:

        compressed_results.append(
            compress_result(r)
        )

    return json.dumps(
        compressed_results,
        indent=2
    )

In [0]:
def generate_llm_response(
    query,
    retrieved_results
):

    context = build_small_context(
        retrieved_results
    )

    prompt = f"""
You are a Healthcare Infrastructure AI.

Answer the user's question using the
facility information provided.

USER QUESTION:
{query}

FACILITY DATA:
{context}

RULES:
- Be concise
- Mention weak infrastructure
- Mention contradictions if important
- Avoid hallucinations
- Keep answer under 250 words
"""

    response = llm.invoke(prompt)

    return response.content

In [0]:
def healthcare_idp_agent(query):

    query_lower = query.lower()

    # --------------------------------
    # MEDICAL DESERTS
    # --------------------------------

    if "medical desert" in query_lower:

        deserts = detect_medical_deserts()

        limited_deserts = deserts[:10]

        prompt = f"""
        Summarize these underserved regions.

        DATA:
        {json.dumps(limited_deserts, indent=2)}

        Keep response concise.
        """

        response = llm.invoke(prompt)

        return response.content

    # --------------------------------
    # WEAK CLINICS
    # --------------------------------

    elif "weak clinic" in query_lower:

        weak_clinics = find_weak_clinics()

        limited_weak = weak_clinics[:10]

        prompt = f"""
        Analyze these weak clinics.

        DATA:
        {json.dumps(limited_weak, indent=2)}

        Explain the common weaknesses.
        Keep response under 200 words.
        """

        response = llm.invoke(prompt)

        return response.content

    # --------------------------------
    # GENERAL QUERIES
    # --------------------------------

    else:

        retrieved_results = (
            retrieve_facilities(
                query,
                k=3
            )
        )

        return generate_llm_response(
            query,
            retrieved_results
        )

In [0]:
response = healthcare_idp_agent(
    "Show hospitals with missing ICU capability"
)

print(response)

Based on the provided facility data, the following hospitals have missing ICU capability:

1. **Nk-Salem Medical Centre** (Accra) - classified as Weak, with very low doctor count and no mention of ICU capability.
2. **Aneeja Hospital** (Accra) - classified as Weak, with very low doctor count, small inpatient facility, and no mention of ICU capability.

These facilities have weak infrastructure, which may indicate a lack of resources or capacity to provide critical care services. It is essential to note that the absence of ICU capability may lead to inadequate care for patients requiring intensive treatment.

Additionally, **Bemuah Royal Hospital** (Accra) has a contradiction in its classification as Moderate, despite having an ICU capability detected. This contradiction may indicate a discrepancy in the facility's assessment.


In [0]:
response = healthcare_idp_agent(
    "Find weak clinics with poor infrastructure"
)

print(response)

The common weaknesses among the clinics are:

1. **Very low doctor count**: This is the most common reason for a clinic being classified as weak, appearing in 9 out of 10 clinics.
2. **Facility classified as Weak**: This is a given, as the data is specifically about weak clinics.
3. **Official website available**: While having an official website is a positive aspect, it is also mentioned as a reason for a clinic being weak in 2 out of 10 clinics.

The clinics with multiple specialties available (appearing in 1 out of 10 clinics) do not seem to be a common weakness. However, it is worth noting that the presence of multiple specialties might not necessarily be a weakness, but rather a strength.

Overall, the data suggests that the primary concern for these weak clinics is the low number of doctors available.


In [0]:
response = healthcare_idp_agent(
    "Which hospitals may have suspicious cardiology claims?"
)

print(response)

Based on the provided facility data, the following hospitals may have suspicious cardiology claims due to weak infrastructure:

1. **Korle Bu Teaching Hospital** (Accra) - Score: 12, Category: Weak. Although it has advanced equipment (CT), the very low doctor count raises concerns.
2. **Precise Specialist Clinic** (Kumasi) - Score: 8, Category: Weak. The very low doctor count is a significant issue.
3. **Yaaba Medical Services** (Accra) - Score: 6, Category: Weak. Similar to the previous two, the very low doctor count is a major concern.

It's essential to note that these facilities have been classified as Weak due to various reasons, including low doctor counts. This may indicate potential issues with the quality of care provided, which could lead to suspicious claims. However, it's crucial to investigate further to confirm any suspicions.

There are no contradictions in the provided data that would affect the analysis.


In [0]:
response = healthcare_idp_agent(
    "Which regions appear to be medical deserts?"
)

print(response)

The data represents underserved regions with limited ICU (Intensive Care Unit) capabilities. 

Key findings:

- There are 64 facilities without ICU capability in an unspecified city.
- Accra has 2 facilities without ICU capability.
- Other cities with limited ICU capabilities include:
  - Agroyesum (1 facility)
  - Anloga (1 facility)
  - Abelenkpe, Accra (1 facility)
  - Abesim (1 facility)
  - Abesim - Sunyani (1 facility)
  - Aboadze (1 facility)
  - Abomosu (1 facility)
  - Abuakwa (1 facility)


In [0]:
response = healthcare_idp_agent(
    "Find facilities claiming trauma care without blood bank support"
)

print(response)

Based on the provided facility data, I found that the following facilities claim to provide trauma care without blood bank support:

1. **Rescue Clinic**: This facility is classified as Weak, indicating a potential weak infrastructure. It has a very low doctor count, which may impact its ability to provide adequate trauma care. There are no contradictions in its data.

2. **The Bank Hospital (Accra)**: This facility is also classified as Weak, indicating a potential weak infrastructure. It has a very low doctor count, which may impact its ability to provide adequate trauma care. However, it has a contradiction in its data - it claims to have multiple specialties available, but its overall score and category suggest otherwise.

Please note that these facilities may not have the necessary resources or infrastructure to provide effective trauma care. It is essential to verify this information with the facilities directly to ensure accuracy.


In [0]:
def calculate_result_confidence(result):

    strength = result.get(
        "strength",
        {}
    )

    score = strength.get("score", 0)

    contradictions = result.get(
        "contradictions",
        []
    )

    penalty = len(
        contradictions
    ) * 8

    confidence = max(
        0,
        min(
            100,
            score - penalty
        )
    )

    return round(confidence, 2)

In [0]:
def healthcare_idp_agent_v2(query):

    results = retrieve_facilities(
        query,
        k=2
    )

    compact_results = []

    for r in results:

        compact = compress_result(r)

        compact["confidence"] = (
            calculate_result_confidence(r)
        )

        compact_results.append(compact)

    prompt = f"""
You are an expert healthcare infrastructure
analysis assistant.

USER QUERY:
{query}

HEALTHCARE DATA:
{json.dumps(compact_results[:2], indent=2)}

INSTRUCTIONS:
- Answer clearly and professionally
- Mention infrastructure gaps
- Identify suspicious inconsistencies
- Explain confidence briefly
- Keep response under 150 words
- Use bullet points when helpful
"""

    response = llm.invoke(prompt)

    return response.content

In [0]:
query = """
Which facilities may have advanced
specialties without enough
supporting infrastructure?
"""

response = healthcare_idp_agent_v2(query)

print(response)

**Facilities with Advanced Specialties and Infrastructure Gaps**

Based on the provided healthcare data, the following facilities may have advanced specialties without enough supporting infrastructure:

* **Greater Accra Regional Hospital**:
 + Multiple specialties available
 + Very low doctor count
 + Official website available
* **HealthLink Hospital**:
 + Multiple specialties available
 + Very low doctor count
 + Advanced equipment detected (CT, dialysis)

**Suspicious Inconsistencies:**

* Both facilities are classified as "Weak" despite having advanced equipment and multiple specialties.
* Low doctor count may indicate insufficient staffing to support advanced specialties.

**Confidence:** The confidence level for these findings is 9 out of 10, indicating a high degree of certainty based on the available data.


In [0]:
df.to_csv(
    "/Volumes/workspace/default/my_volume/final_healthcare_agent_dataset.csv",
    index=False
)

print("Final dataset saved successfully.")

Final dataset saved successfully.


In [0]:
demo_queries = [

    "Show hospitals with missing ICU capability",

    "Find weak clinics",

    "Which regions are medical deserts?",

    "Find suspicious cardiology claims",

    "Which facilities may lack proper trauma infrastructure?"
]

for q in demo_queries:

    print("=" * 80)
    print("QUERY:")
    print(q)

    print("\nRESPONSE:\n")

    response = healthcare_idp_agent_v2(q)

    print(response)

    print("\n")

QUERY:
Show hospitals with missing ICU capability

RESPONSE:

**Hospitals with Missing ICU Capability:**

Based on the provided healthcare data, the following hospitals are identified as having missing ICU capability:

* Bemuah Royal Hospital (Accra) - Although ICU capability was detected, it is listed as a top reason for the facility's Moderate classification, indicating potential inconsistencies.

**Infrastructure Gaps:**

* Inadequate ICU capacity may lead to insufficient care for critically ill patients.
* Limited healthcare resources may exacerbate existing healthcare challenges.

**Suspicious Inconsistencies:**

* Bemuah Royal Hospital's contradictory information regarding ICU capability.

**Confidence:**

The confidence score represents the reliability of the data, with higher scores indicating more accurate information. In this case, the confidence score for Bemuah Royal Hospital is 40, suggesting some uncertainty in the data.


QUERY:
Find weak clinics

RESPONSE:

**Weak Clini

In [0]:
# ============================================
# INTERACTIVE HEALTHCARE IDP AGENT DEMO
# ============================================

from IPython.display import display, Markdown

def run_healthcare_demo():

    print("\n" + "="*80)
    print("🏥 Healthcare Infrastructure IDP Agent")
    print("="*80)

    print("\nExample Queries:")
    print("• Show hospitals with missing ICU capability")
    print("• Find weak clinics")
    print("• Which regions are medical deserts?")
    print("• Find suspicious cardiology claims")
    print("• Which facilities may lack trauma infrastructure?")
    print("\nType 'exit' to stop.\n")

    while True:

        user_query = input("🔍 Enter your healthcare query: ")

        if user_query.lower() in ["exit", "quit", "q"]:
            print("\n✅ Session Ended.")
            break

        if len(user_query.strip()) == 0:
            print("\n⚠️ Please enter a valid query.\n")
            continue

        try:

            print("\n⏳ Analyzing healthcare infrastructure...\n")

            response = healthcare_idp_agent_v2(user_query)

            display(Markdown(
                f"""
## 🧠 Agent Response

{response}
"""
            ))

            print("\n" + "-"*80 + "\n")

        except Exception as e:

            print("\n❌ Error occurred:")
            print(str(e))
            print("\nTry another query.\n")


# Run Demo
run_healthcare_demo()


🏥 Healthcare Infrastructure IDP Agent

Example Queries:
• Show hospitals with missing ICU capability
• Find weak clinics
• Which regions are medical deserts?
• Find suspicious cardiology claims
• Which facilities may lack trauma infrastructure?

Type 'exit' to stop.



🔍 Enter your healthcare query:  find weak cliics


⏳ Analyzing healthcare infrastructure...




## 🧠 Agent Response

**Weak Clinics Analysis**

Based on the provided healthcare data, the following clinics have been identified as weak:

* COA Research & Manufacturing (Cape Coast) - Score: 7, Category: Weak
* Alma Medical Laboratory (Accra) - Score: 2, Category: Weak

**Infrastructure Gaps:**

* Very low doctor count at both facilities
* Official website available at COA Research & Manufacturing, but not at Alma Medical Laboratory

**Suspicious Inconsistencies:**

* None identified

**Confidence:**
The confidence score represents the level of certainty in the classification of a facility as weak. A higher score indicates a stronger confidence in the classification.

**Recommendations:**
Addressing the very low doctor count and improving infrastructure at these facilities is crucial to enhance their overall performance and provide better healthcare services to patients.



--------------------------------------------------------------------------------



...


⏳ Analyzing healthcare infrastructure...




## 🧠 Agent Response

**Medical Deserts Analysis**

Based on the provided dataset, the following examples highlight medical deserts in Ghana:

* **Dunkwa Municipal Hospital**: Classified as "Weak" with a score of 4, due to a very low doctor count. This indicates a significant infrastructure gap in the Dunkwa-On-Offin area.
* **GPHA Clinic**: Also classified as "Weak" with a score of 8, but with a potential contradiction: trauma services without a blood bank. This raises suspicions about the clinic's ability to provide adequate care.

**Infrastructure Gaps:**

* Low doctor count in Dunkwa-On-Offin
* Potential lack of blood bank services in Takoradi

**Suspicious Inconsistencies:**

* GPHA Clinic's high score (8) despite being classified as "Weak"
* Low confidence score (0) for GPHA Clinic, indicating potential data inconsistencies

**Confidence:** The confidence score represents the model's certainty in its classification. A score of 4 indicates moderate confidence, while a score of 0 suggests low confidence due to potential data inconsistencies.



--------------------------------------------------------------------------------



🔍 Enter your healthcare query:  exit


✅ Session Ended.
